Validation of CCSD-seeded LUCJ (../hamiltonian/H2O_CCSD_seeded_LUCJ_instructions.md), for the H2O CAS(4e,4o) system.
 
This script initially implements Part 1 exactly, then runs a set of diagnostic checks (Part 2) that verify if the CCSD-seeded state reproduces the claimed CCSD energy, without changing the Hamiltonian or the SQD post-processing at all. Only the state-preparation step is touched.

In [24]:
%pip install ffsim

In [25]:
from itertools import combinations
 
import numpy as np
import ffsim
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

In [26]:
import os

if not os.path.exists("SQD"):
    !git clone https://github.com/KhushiPandey1805/SQD.git SQD
    print("Cloned fresh.")
else:
    os.chdir("SQD")
    !git pull
    os.chdir("/content")
    print("Already present, pulled latest changes.")

!ls /content/SQD/code

import sys
sys.path.insert(0, "/content/SQD/code")

from hamiltonianCode import (
    interleaved_to_blocked_layout,
    parse_hamiltonian_file,
    parse_metadata,
)


Already up to date.
Already present, pulled latest changes.
hamiltonianCode.ipynb  N2ex.ipynb   simVShardware.ipynb
hamiltonianCode.py     __pycache__


In [27]:
HAMILTONIAN_FILEPATH = "/content/SQD/hamiltonian/H2O_CAS4e4o_JW_Pauli_Hamiltonian_HERMITIAN.txt"
AMPLITUDES_FILEPATH = "/content/SQD/hamiltonian/H2O_CAS4e4o_CCSD_t_amplitudes.npz"

In [28]:
def load_everything():
    hamiltonian = parse_hamiltonian_file(HAMILTONIAN_FILEPATH)
    meta = parse_metadata(HAMILTONIAN_FILEPATH)
    norb = meta["n_spatial_orbitals"]
    nelec = (meta["n_alpha"], meta["n_beta"])
 
    # Interleaved -> spin-blocked [0a,1a,2a,3a,0b,1b,2b,3b], as required by
    # UCJOpSpinBalancedJW / PrepareHartreeFockJW (per the instructions file).
    layout = interleaved_to_blocked_layout(norb)
    hamiltonian_blocked = hamiltonian.apply_layout(layout, num_qubits=2 * norb).simplify()
 
    amps = np.load(AMPLITUDES_FILEPATH)
    return hamiltonian_blocked, meta, norb, nelec, amps

In [29]:
def build_circuit(norb, nelec, lucj_op):
    qubits = QuantumRegister(2 * norb, name="q")
    circuit = QuantumCircuit(qubits)
    circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)
    circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(lucj_op), qubits)
    return circuit

In [30]:
def noiseless_energy(circuit, hamiltonian):
    estimator = StatevectorEstimator()
    return float(estimator.run([(circuit, hamiltonian)]).result()[0].data.evs)

In [31]:
def particle_number_fraction(circuit, norb, n_alpha, n_beta, shots=2000, seed=0):
    """Fraction of sampled shots with the correct N_alpha=N_beta count,
    for spin-blocked ordering [0a,...,(norb-1)a, 0b,...,(norb-1)b]."""
    circuit = circuit.copy()
    circuit.measure_all()
    sampler = StatevectorSampler(seed=np.random.default_rng(seed))
    job = sampler.run([circuit], shots=shots)
    bitstrings = job.result()[0].data.meas.get_bitstrings()
 
    def is_valid(bs):
        # Qiskit bitstrings read qubit(N-1)...qubit0 left to right.
        alpha_count = sum(int(b) for b in bs[len(bs) - norb:])       # qubits 0..norb-1
        beta_count = sum(int(b) for b in bs[:len(bs) - norb])         # qubits norb..2norb-1
        return alpha_count == n_alpha and beta_count == n_beta
 
    return sum(is_valid(b) for b in bitstrings) / len(bitstrings)

In [32]:
def part1(hamiltonian, meta, norb, nelec, amps):
    print("=" * 70)
    print("PART 1: (n_reps=1, optimize=True)")
    print("=" * 70)
 
    t1, t2 = amps["t1"], amps["t2"]
    pairs_aa = [(p, p + 1) for p in range(norb - 1)]
    pairs_ab = [(p, p) for p in range(norb)]
 
    lucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
        t2=t2, t1=t1, n_reps=1,
        interaction_pairs=(pairs_aa, pairs_ab),
        optimize=True, options={"maxiter": 1000},
    )
    circuit = build_circuit(norb, nelec, lucj_op)
 
    energy = noiseless_energy(circuit, hamiltonian)
    error_mha = (energy - meta["reference_energy"]) * 1000
    print(f"CCSD-seeded LUCJ noiseless energy: {energy:.6f} Ha  (error {error_mha:.4f} mHa)")
    print(f"Amplitude file's own claimed CCSD energy: {amps['ccsd_energy']:.6f} Ha "
          f"(error {(amps['ccsd_energy'] - meta['reference_energy']) * 1000:.4f} mHa)")
 
    frac = particle_number_fraction(circuit, norb, meta["n_alpha"], meta["n_beta"])
    print(f"Fraction of shots with correct N_alpha=N_beta: {frac:.4f}")
 
    # Compare against a randomly-initialized LUCJ of the same n_reps, as a
    # sanity check that CCSD seeding is at least doing SOMETHING useful.
    rng = np.random.default_rng(1)
    random_op = ffsim.random.random_ucj_op_spin_balanced(norb, n_reps=1, seed=1)
    random_circuit = build_circuit(norb, nelec, random_op)
    random_energy = noiseless_energy(random_circuit, hamiltonian)
    print(f"For comparison, a RANDOM n_reps=1 LUCJ gives: {random_energy:.6f} Ha  "
          f"(error {(random_energy - meta['reference_energy']) * 1000:.4f} mHa)")
 
    return energy

In [33]:
def part2_diagnostics(hamiltonian, meta, norb, nelec, amps):
    print()
    print("=" * 70)
    print("PART 2: Diagnostics -- isolating the source of the discrepancy")
    print("=" * 70)
    t1, t2 = amps["t1"], amps["t2"]
    reference_energy = meta["reference_energy"]
 
    def test(t1_use, t2_use, n_reps, interaction_pairs, label):
        lucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
            t2=t2_use, t1=t1_use, n_reps=n_reps, interaction_pairs=interaction_pairs)
        circuit = build_circuit(norb, nelec, lucj_op)
        energy = noiseless_energy(circuit, hamiltonian)
        error_mha = (energy - reference_energy) * 1000
        print(f"  {label}: energy={energy:.6f}  error={error_mha:.4f} mHa")
        return energy
 
    pairs_aa = [(p, p + 1) for p in range(norb - 1)]
    pairs_ab = [(p, p) for p in range(norb)]
 
    print("\n-- Control: zero amplitudes should EXACTLY reproduce Hartree-Fock --")
    zero_energy = test(np.zeros_like(t1), np.zeros_like(t2), 1, (pairs_aa, pairs_ab),
                        "zero amplitudes")
    hf_circuit = build_circuit(
        norb, nelec, ffsim.UCJOpSpinBalanced.from_t_amplitudes(
            t2=np.zeros_like(t2), t1=np.zeros_like(t1), n_reps=1,
            interaction_pairs=(pairs_aa, pairs_ab)))
    print(f"  (compare to data file's own reference_energy: {amps['reference_energy']:.6f})")
    if abs(zero_energy - amps["reference_energy"]) > 1e-4:
        print("  *** FAIL: even the zero-amplitude control doesn't match HF! "
              "Something is wrong with the circuit-building itself. ***")
    else:
        print("  PASS: matches HF reference -- circuit construction is correct.")
 
    print("\n-- Does increasing n_reps help? (rules out 'too few terms') --")
    for n_reps in [1, 2, 3, 4]:
        test(t1, t2, n_reps, (pairs_aa, pairs_ab), f"n_reps={n_reps}")
 
    print("\n-- Full exact decomposition, no truncation, no pair restriction --")
    test(t1, t2, None, None, "n_reps=None, unrestricted")
 
    print("\n-- Sign / transpose convention checks --")
    test(-t1, -t2, None, None, "sign-flipped (both negated)")
    test(t1, np.transpose(t2, (0, 1, 3, 2)), None, None, "t2 with a,b transposed")
    test(t1, np.transpose(t2, (1, 0, 2, 3)), None, None, "t2 with i,j transposed")
    test(t1.T, t2, None, None, "t1 transposed")
 
    print("\n-- Isolating t1 vs t2 contributions --")
    test(np.zeros_like(t1), t2, None, None, "t1=0, only t2")
    test(t1, np.zeros_like(t2), None, None, "t2=0, only t1")
 
    print()
    print("If NONE of the above get within ~1 mHa of the claimed CCSD energy")
    print(f"({amps['ccsd_energy']:.6f} Ha, i.e. error {(amps['ccsd_energy']-reference_energy)*1000:.4f} mHa),")
    print("this points to a mismatch between the Hamiltonian file's and the")
    print("amplitude file's spatial-orbital ordering/labeling convention --")
    print("not a bug in n_reps, interaction_pairs, or sign convention.")

In [34]:
if __name__ == "__main__":
    hamiltonian, meta, norb, nelec, amps = load_everything()
    print(f"Loaded: {hamiltonian.num_qubits} qubits, nelec={nelec}, "
          f"exact reference energy={meta['reference_energy']:.6f}\n")
 
    part1(hamiltonian, meta, norb, nelec, amps)
    part2_diagnostics(hamiltonian, meta, norb, nelec, amps)

Loaded: 8 qubits, nelec=(2, 2), exact reference energy=-76.116691

PART 1: (n_reps=1, optimize=True)
CCSD-seeded LUCJ noiseless energy: -76.056149 Ha  (error 60.5415 mHa)
Amplitude file's own claimed CCSD energy: -76.116597 Ha (error 0.0939 mHa)
Fraction of shots with correct N_alpha=N_beta: 1.0000
For comparison, a RANDOM n_reps=1 LUCJ gives: -75.127695 Ha  (error 988.9963 mHa)

PART 2: Diagnostics -- isolating the source of the discrepancy

-- Control: zero amplitudes should EXACTLY reproduce Hartree-Fock --
  zero amplitudes: energy=-76.063572  error=53.1193 mHa
  (compare to data file's own reference_energy: -76.063572)
  PASS: matches HF reference -- circuit construction is correct.

-- Does increasing n_reps help? (rules out 'too few terms') --
  n_reps=1: energy=-76.067089  error=49.6018 mHa
  n_reps=2: energy=-76.066867  error=49.8236 mHa
  n_reps=3: energy=-76.071705  error=44.9858 mHa
  n_reps=4: energy=-76.075755  error=40.9363 mHa

-- Full exact decomposition, no truncation